<a href="https://colab.research.google.com/github/mscampb4-ncsu/Homework6/blob/main/Homework6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST 554 - Homework 6
Author: Max Campbell

Please note that, since the first part of this assignment uses the same database from the previous homework assignment, the previous part is included here. The questions directly relevant to Homework 6 will be discussed under **Part I - (More Practice) Querying a Database**.

## Querying a Database

Now, let's do some exploratory analysis on the baseball archive data mentioned previously using SQL! First, we will read in the data and see what we have to work with.



In [2]:
#Connect to the database file
import sqlite3
import pandas as pd

current_input = '''
  SELECT *
  FROM sqlite_schema
  WHERE type = "table";
'''

con = sqlite3.connect("lahman_1871-2022.sqlite")
schema = pd.read_sql(current_input, con)
schema

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


The first thing we want to look at is the `Teams` table. Specifically, we want to see which teams played in 2015.

In [3]:
#Get all teams that played in 2015
current_input = '''
  SELECT *
  FROM Teams
  WHERE yearID = '2015'
  '''

output = pd.read_sql(current_input, con)
output

,yearID,lgID,teamID,franchID,divID,Rank,G,Ghome,W,L,...,DP,FP,name,park,attendance,BPF,PPF,teamIDBR,teamIDlahman45,teamIDretro
0,2015,AL,BAL,BAL,E,3,162,78,81,81,...,134,0.987,Baltimore Orioles,Oriole Park at Camden Yards,2281202,103,104,BAL,BAL,BAL
1,2015,AL,BOS,BOS,E,5,162,81,78,84,...,148,0.984,Boston Red Sox,Fenway Park II,2880694,104,107,BOS,BOS,BOS
2,2015,AL,CHA,CHW,C,4,162,81,76,86,...,159,0.983,Chicago White Sox,U.S. Cellular Field,1755810,92,93,CHW,CHA,CHA
3,2015,AL,CLE,CLE,C,3,161,80,81,80,...,136,0.987,Cleveland Indians,Progressive Field,1388905,106,106,CLE,CLE,CLE
4,2015,AL,DET,DET,C,5,161,81,74,87,...,165,0.986,Detroit Tigers,Comerica Park,2726048,97,98,DET,DET,DET
5,2015,AL,HOU,HOU,W,2,162,81,86,76,...,131,0.986,Houston Astros,Minute Maid Park,2153585,97,99,HOU,HOU,HOU
6,2015,AL,KCA,KCR,C,1,162,81,95,67,...,138,0.985,Kansas City Royals,Kauffman Stadium,2708549,104,103,KCR,KCA,KCA
7,2015,AL,LAA,ANA,W,3,162,81,85,77,...,108,0.984,Los Angeles Angels of Anaheim,Angel Stadium of Anaheim,3012765,94,95,LAA,ANA,ANA
8,2015,AL,MIN,MIN,C,2,162,81,83,79,...,150,0.986,Minnesota Twins,Target Field,2220054,103,104,MIN,MIN,MIN
9,2015,AL,NYA,NYY,E,2,162,81,87,75,...,135,0.985,New York Yankees,Yankee Stadium III,3193795,99,101,NYY,NYA,NYA


Next, let's look at some information about players in the Hall of Fame. Specifically, we want to know what year they were inducted and for what reason.

In [15]:
#Get Hall of Famers
current_input = '''
  SELECT playerID, yearID, category
  FROM HallOfFame
  WHERE inducted = 'Y'
  '''

output = pd.read_sql(current_input, con)
output

,playerID,yearid,category
0,aaronha01,1982,Player
1,alexape01,1938,Player
2,alomaro01,2011,Player
3,alstowa01,1983,Manager
4,andersp01,2000,Manager
...,...,...,...
335,yastrca01,1989,Player
336,yawketo99,1980,Pioneer/Executive
337,youngcy01,1937,Player
338,youngro01,1972,Player


Next, we want to know how many different people have managed the Pittsburgh Pirates over the years.

In [5]:
#Get managers of the Pittsburgh Pirates
current_input = '''
  SELECT DISTINCT playerID
  FROM Managers
  WHERE teamID = 'PIT'
  '''

output = pd.read_sql(current_input, con)
output

,playerID
0,bezdehu99
1,bragabo01
2,buckeal99
3,burnsto01
4,burwebi01
5,bushdo01
6,callani01
7,clarkfr01
8,davissp01
9,donovpa01


Looks like there's been 41 managers. Now, let's try and figure out how many of these people where inducted into the hall of fame!

In [6]:
#Get managers who are Hall of Famers
current_input = '''
  SELECT DISTINCT h.playerID
  FROM HallOfFame AS h
  INNER JOIN Managers AS m
    ON h.playerID = m.playerID
  WHERE h.inducted = 'Y'
  '''

output = pd.read_sql(current_input, con)
print("Number of people who managed a team that are hall of famers:", output.size)

Number of people who managed a team that are hall of famers: 97


Now, we want to now how these managers performed! We will get the wins, losses, and games managed per season for each of the 97 managers above!

In [7]:
#Get W-L records for managers who are hall of famers
current_input = '''
  SELECT m.playerID, m.G, m.W, m.L
  FROM Managers AS m
  INNER JOIN HallOfFame AS h
    ON m.playerID = h.playerID
  WHERE h.inducted = 'Y'
  '''

output = pd.read_sql(current_input, con)
output

,playerID,G,W,L
0,alstowa01,154,92,62
1,alstowa01,154,98,55
2,alstowa01,154,93,61
3,alstowa01,154,84,70
4,alstowa01,154,71,83
...,...,...,...,...
991,wrighha01,46,22,23
992,wrighha01,138,68,69
993,wrighha01,155,87,66
994,wrighha01,133,72,57


Those are some impressive records! Now, let's see how their overall records compare. First, we will calculate the overall wins and losses for each manager. Then, we will calculate their winning percentage as `Wins / (Wins + Losses)`, and then finally we will sort by the highest to lowest winning percentage to see who was most successful!

In [8]:
#Get W-L records for managers who are hall of famers
current_input = '''
  SELECT
    m.playerID,
    SUM(m.G) as G,
    SUM(m.W) as W,
    SUM(m.L) as L,
    (CAST(SUM(m.W) as FLOAT) / (SUM(m.W) + SUM(m.L))) as WLPercent
  FROM Managers AS m
  INNER JOIN HallOfFame AS h
    ON m.playerID = h.playerID
  WHERE h.inducted = 'Y'
  GROUP BY m.playerID
  ORDER BY WLPercent DESC
  '''

output = pd.read_sql(current_input, con)
output

,playerID,G,W,L,WLPercent
0,simmote01,1,1,0,1.000000
1,wrighge01,85,59,25,0.702381
2,spaldal01,126,78,47,0.624000
3,mccarjo99,3487,2125,1333,0.614517
4,comisch01,1410,840,541,0.608255
...,...,...,...,...,...
92,bottoji01,78,21,56,0.272727
93,applilu01,40,10,30,0.250000
94,baineha01,4,1,3,0.250000
95,wagneho01,5,1,4,0.200000


Putting aside the outlier who won the only game they managed, it looks like `wrighge01` (otherwise known as George Wright), was the manager with the highest winning percentage. Baseball Reference shows that this was accomplished in the [1879 season with the Providence Grays](https://www.baseball-reference.com/managers/wrighge01.shtml). Note that the manager wins column was cast as a float in the numerator of the operation to derive `WLPercent` to avoid any loss of decimal precision in the resulting output.

## Part I - (More Practice) Querying a Database

For this part, we aim to create a table of basic statistics for hall of famers that were also pitchers. Given that we want this to be comprehensive, we also want to make sure that the basic batting statistics are included in this table. To begin, let's revist the schema of this database:

In [9]:
#Re-visit schema from previous part
current_input = '''
  SELECT *
  FROM sqlite_schema
  WHERE type = "table";
'''

con = sqlite3.connect("lahman_1871-2022.sqlite")
schema = pd.read_sql(current_input, con)
schema

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


Now, let's join the `HallofFame` and `Pitching` tables to see which hall of famers were pitchers and what their pitching stats looked like.

In [24]:
#Get Hall of Fame Pitching stats
current_input = '''
  SELECT
    p.playerID,
    SUM(p.GS) as GS,
    SUM(p.G) as G,
    SUM(p.W) as W,
    SUM(p.L) as L,
    SUM(p.IPOuts) as IPOuts,
    SUM(p.CG) as CG,
    SUM(p.SHO) as SHO,
    SUM(p.SV) as SV
  FROM Pitching AS p
  INNER JOIN HallOfFame AS h
    ON p.playerID = h.playerID
  WHERE h.inducted = 'Y'
  GROUP BY p.playerID;
'''

output = pd.read_sql(current_input, con)
output

,playerID,GS,G,W,L,IPOuts,CG,SHO,SV
0,alexape01,599,696,373,208,15570,437,90,32
1,ansonca01,0,3,0,1,12,0,0,1
2,becklja01,1,1,0,1,12,0,0,0
3,bendech01,334,459,212,127,9051,255,40,34
4,blylebe01,685,692,287,250,14910,242,60,0
...,...,...,...,...,...,...,...,...,...
103,willivi01,471,513,249,205,11988,388,50,11
104,wrighge01,0,3,0,1,15,0,0,0
105,wrighha01,8,36,4,4,301,0,0,14
106,wynnea01,612,691,300,244,13692,290,49,15


Looks like there were 108 hall of famers who pitched. Just by looking at the high-level view of the table, we can see that there are a few pitchers who only started a handful of games as a pitcher, and may have mainly played batting positions. As we said before, getting the batting statistics for these pitchers is important for getting a comprehensive view of their impact on baseball games, so let's pull those statistics as well.

In [41]:
#Grab Hall of fame pitcher batting statistics.
current_input = '''
  SELECT
    b.playerID,
    SUM(b.AB) as AB,
    SUM(b.R) as R,
    SUM(b.H) as H,
    SUM(b.HR) as HR,
    SUM(b.RBI) as RBI,
    SUM(b.BB) as BB,
    SUM(b.SO) as SO
  FROM Batting AS b
  INNER JOIN (SELECT DISTINCT playerID FROM Pitching) AS p
    ON b.playerID = p.playerID
  INNER JOIN (SELECT playerID, inducted FROM HallOfFame) AS h
    ON b.playerID = h.playerID
  WHERE h.inducted = 'Y'
  GROUP BY b.playerID;
'''

output = pd.read_sql(current_input, con)
output

,playerID,AB,R,H,HR,RBI,BB,SO
0,alexape01,1810,154,378,11,163,77,276
1,ansonca01,10281,1999,3435,97,2075,984,330
2,becklja01,9551,1603,2938,87,1581,616,526
3,bendech01,1147,102,243,6,116,75,143
4,blylebe01,451,19,59,0,25,5,193
...,...,...,...,...,...,...,...,...
103,willivi01,1493,107,248,1,84,81,199
104,wrighge01,2873,665,866,11,326,68,119
105,wrighha01,813,183,224,4,113,37,14
106,wynnea01,1704,136,365,17,173,141,330


Judging from this dataset and the previous one that we pulled, it is becoming increasingly apparent why the MLB named their award for the best pitching performance after Cy Young (whose statistics are in row 107). We have all the data that we wanted, but we have it across two separate tables, which makes it harder to read. Thankfully, both tables cover the same set of players, so combining them should be fairly straightforward:

In [67]:
#Combine the pitching and batting tables pulled above
current_input = '''
  SELECT
    p.playerID,
    SUM(p.GS) as GS,
    SUM(p.G) as G,
    SUM(p.W) as W,
    SUM(p.L) as L,
    SUM(p.IPOuts) as IPOuts,
    SUM(p.CG) as CG,
    SUM(p.SHO) as SHO,
    SUM(p.SV) as SV,
    AB,
    b.R,
    b.H,
    b.HR,
    RBI,
    b.BB,
    b.SO
  FROM Pitching AS p
  INNER JOIN
    (SELECT DISTINCT playerID,
    SUM(AB) as AB,
    SUM(R) as R,
    SUM(H) as H,
    SUM(HR) as HR,
    SUM(RBI) as RBI,
    SUM(BB) as BB,
    SUM(SO) as SO
    FROM Batting
    GROUP BY playerID) AS b
    ON p.playerID = b.playerID
  INNER JOIN (SELECT playerID, inducted FROM HallOfFame) AS h
    ON p.playerID = h.playerID
  WHERE h.inducted = 'Y'
  GROUP BY p.playerID;
'''

output = pd.read_sql(current_input, con)
output

,playerID,GS,G,W,L,IPOuts,CG,SHO,SV,AB,R,H,HR,RBI,BB,SO
0,alexape01,599,696,373,208,15570,437,90,32,1810,154,378,11,163,77,276
1,ansonca01,0,3,0,1,12,0,0,1,10281,1999,3435,97,2075,984,330
2,becklja01,1,1,0,1,12,0,0,0,9551,1603,2938,87,1581,616,526
3,bendech01,334,459,212,127,9051,255,40,34,1147,102,243,6,116,75,143
4,blylebe01,685,692,287,250,14910,242,60,0,451,19,59,0,25,5,193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,willivi01,471,513,249,205,11988,388,50,11,1493,107,248,1,84,81,199
104,wrighge01,0,3,0,1,15,0,0,0,2873,665,866,11,326,68,119
105,wrighha01,8,36,4,4,301,0,0,14,813,183,224,4,113,37,14
106,wynnea01,612,691,300,244,13692,290,49,15,1704,136,365,17,173,141,330


Now we can see everything we were looking for in one place!